# Iris Classification Demo

This notebook demonstrates `LeafEncoder` and `KernelClassifier` on the Iris dataset using the `src/` package layout.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find the project root containing pyproject.toml and src/.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.model_selection import train_test_split

from forestkernel import LeafEncoder
from forestkernel.prediction import KernelClassifier
from forestkernel.prediction.functional import kernel_predict


In [ ]:
seed = 42
forest_type = "rf"  # choose: "rf", "et", "gbt", "xgb", or "lgbm"
weight_scheme = "gap"  # rf/et: "uniform", "kerf", "oob", "gap"; boosted: "uniform", "kerf", "boosted"

def make_classifier(kind, *, random_state, n_estimators=100):
    if kind == "rf":
        return RandomForestClassifier(
            n_estimators=n_estimators,
            bootstrap=True,
            oob_score=True,
            n_jobs=-1,
            random_state=random_state,
        )
    if kind == "et":
        return ExtraTreesClassifier(
            n_estimators=n_estimators,
            bootstrap=True,
            oob_score=True,
            n_jobs=-1,
            random_state=random_state,
        )
    if kind == "gbt":
        return GradientBoostingClassifier(
            n_estimators=n_estimators,
            random_state=random_state,
        )
    if kind == "xgb":
        from xgboost import XGBClassifier

        return XGBClassifier(
            n_estimators=n_estimators,
            eval_metric="logloss",
            n_jobs=-1,
            random_state=random_state,
            verbosity=0,
        )
    if kind == "lgbm":
        from lightgbm import LGBMClassifier

        return LGBMClassifier(
            n_estimators=n_estimators,
            n_jobs=-1,
            random_state=random_state,
            verbose=-1,
        )
    raise ValueError(f"Unknown forest_type={kind!r}.")

if forest_type in {"gbt", "xgb", "lgbm"} and weight_scheme in {"oob", "gap"}:
    raise ValueError("Boosted estimators do not support OOB or GAP weighting; use uniform, kerf, or boosted.")

forest = make_classifier(forest_type, random_state=seed)


In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.2,
    stratify=iris.target,
    random_state=seed,
)

print(f"Train samples: {X_train.shape[0]}; test samples: {X_test.shape[0]}")
print(f"Features: {list(iris.feature_names)}")


In [ ]:
encoder = LeafEncoder(forest=forest, weight_scheme=weight_scheme).fit(X_train, y_train)

Q_train = encoder.training_query_map()
W_train = encoder.reference_map()
Q_test = encoder.transform(X_test)
K_train = encoder.kernel()
K_test = encoder.kernel_extend(X_test)

print(f"Q_train: {Q_train.shape}, nnz={Q_train.nnz}")
print(f"W_train: {W_train.shape}, nnz={W_train.nnz}")
print(f"Q_test : {Q_test.shape}, nnz={Q_test.nnz}")
print(f"K_train: {K_train.shape}, nnz={K_train.nnz}")
print(f"K_test : {K_test.shape}, nnz={K_test.nnz}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
K_dense = K_train.toarray()
im = ax.imshow(K_dense, cmap="viridis", aspect="equal")
ax.set_xticks([])
ax.set_yticks([])
ax.set_title(f"{weight_scheme.upper()} train-train kernel")
fig.colorbar(im, ax=ax, label="kernel value")
plt.show()

row_sums = np.asarray(K_train.sum(axis=1)).ravel()
print(f"Row sums: min={row_sums.min():.4f}, mean={row_sums.mean():.4f}, max={row_sums.max():.4f}")


In [ ]:
classifier = KernelClassifier(forest=make_classifier(forest_type, random_state=seed), weight_scheme=weight_scheme)
classifier.fit(X_train, y_train)

kernel_pred = classifier.predict(X_test)
forest_pred = classifier.forest_.predict(X_test)
manual_pred = kernel_predict(
    Q=classifier.transform(X_test),
    W=classifier.reference_map(),
    y_train=y_train,
    weight_scheme=weight_scheme,
    prediction_type="classification",
    return_proba=False,
)

print(f"Kernel accuracy: {(kernel_pred == y_test).mean():.3f}")
print(f"Forest accuracy: {(forest_pred == y_test).mean():.3f}")
print(f"Kernel/forest disagreements: {np.count_nonzero(kernel_pred != forest_pred)}")
print(f"Kernel/manual disagreements: {np.count_nonzero(kernel_pred != manual_pred)}")


In [ ]:
trust_scores = classifier.diagnostics.get_test_trust(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(
    X_test[:, 2],
    X_test[:, 3],
    c=y_test,
    s=30 + 120 * (trust_scores.max() - trust_scores),
    cmap="Dark2",
    alpha=0.75,
)
ax.set_xlabel(iris.feature_names[2])
ax.set_ylabel(iris.feature_names[3])
ax.set_title("Test points sized by low trust")
plt.show()
